# Gale Crater map tiles: CTX imagery (Mars Trek) + hillshaded topography (built from DTMs)

Produces two tile layers for the web map, on the **same tile grid and bounding box**, so the UI/UX team can stack them. The imagery goes up to **zoom 12** (about 10 m/pixel) and the topography to **zoom 11** (about 20 m/pixel, the resolution of the DTMs):

| Layer | Source | How it is made |
|---|---|---|
| `imagery` (base) | MRO CTX Gale mosaic, Mars Trek layer `Gale_CTX_BlockAdj_dd` | downloaded tile by tile, zooms 7-12 |
| `topo` (toggle on top) | Gale Crater stereo CTX DTMs, 20 m/pixel (Grindrod & Davis, NHM London, CC BY 4.0) | mosaicked and hillshaded locally (grey only, no colors: the UI team adds those), then cut into tiles, zooms 7-11 |

**Steps**
1. Set the box, zooms and options (cell 2)
2. Helper functions and a self-check (cell 3)
3. Check the Trek product (cell 4)
4. Dry run: tile counts and memory needed (cell 5)
5. Download the CTX imagery tiles (cell 6)
6. Unzip and inspect the DTMs (cell 8)
7. Mosaic the DTMs onto the tile grid and check coverage (cell 9)
8. Grey hillshade, no colors (cell 10)
9. Cut into tiles, build the zoom pyramid, write the manifest (cell 11)
10. Preview both layers (cell 12)
11. Hand `out/` to the UI/UX team (notes at the bottom)

Run on your own machine or Colab. The Claude workspace could not reach `trek.nasa.gov` or install rasterio, so cells 4, 6, 8 and 9 have **not** been run. The hillshade, coloring, tiling and pyramid code (cells 10 and 11) and the tile math were tested here on synthetic data.

Needs: Python 3, `numpy`, `pillow`, `rasterio` (`pip install numpy pillow rasterio`), and `DTMs.zip` from the figshare dataset next to the notebook.

## How the Trek tile grid works

- Geographic (equirectangular) grid, **not** web mercator. Tiles are 256 x 256 px.
- Tile matrix set `default028mm`, style `default`.
- Zoom 0 is 2 columns x 1 row. Each zoom doubles both, so columns = 2^(z+1), rows = 2^z, and one tile spans 180/2^z degrees.
- Origin is the top-left corner (-180 lon, +90 lat); row 0 is the north edge.
- URL: `https://trek.nasa.gov/tiles/Mars/EQ/{product}/1.0.0/{style}/{tms}/{z}/{row}/{col}.{ext}`
- A 404 means no data coverage at that tile, not a failure.
- **Pixel size** near the equator: z7 = 326 m, z8 = 163 m, z9 = 81 m, z10 = 41 m, z11 = 20 m, z12 = 10 m (cell 5 prints this table from the grid).

The `topo` layer is cut on exactly this grid, so a tile `z/row/col` in `imagery` and in `topo` covers the same ground.

Verified from the live capabilities file: `Gale_CTX_BlockAdj_dd` is a `.png` product, style `default`, tile matrix set `default028mm`, zooms 0-12, bbox 135.18 to 139.24 E and 6.57 to 2.81 S.

In [7]:
import glob, json, math, os, time, zipfile
import urllib.request, urllib.error
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor

BASE = 'https://trek.nasa.gov/tiles/Mars/EQ'
OUT = 'out_tiles'
# ONE shared map box for every layer: (lon_min, lat_min, lon_max, lat_max), degrees east / planetocentric north.
# = the area fully covered by the CTX DTMs (136.75-139.25 E, 6.70-4.15 S; read off their coverage map, +/- 0.05 deg)
#   intersected with the Trek CTX mosaic (135.18-139.24 E, 6.57-2.81 S).
# Contains Bradbury Landing (137.44 E, 4.59 S) and the Curiosity traverse points that could be checked.
# It clips the crater rim a little on the west, south and north (crater ~154 km across, centre 137.8 E, 5.4 S).
BBOX = (136.75, -6.57, 139.24, -4.15)

ZMIN = 7             # lowest zoom, shared by both layers
IMAGERY_ZMAX = 12    # CTX imagery goes down to z12 (~10 m/pixel); Trek serves this layer up to z12
TOPO_ZMAX = 11       # topo stops at z11 (~20 m/pixel), the resolution of the DTMs; z12 would only enlarge them
# If cell 6 reports 'no_coverage' for the lowest zooms, Trek has no tiles there: raise ZMIN to 9.

# Layer 1: CTX surface imagery from Mars Trek (the base layer)
LAYERS = {
    'imagery': {'product': 'Gale_CTX_BlockAdj_dd', 'zmin': ZMIN, 'zmax': IMAGERY_ZMAX},
}

# Layer 2: hillshaded topography built locally from the DTMs
TOPO = {
    'zip': 'DTMs.zip',                 # figshare download (DTMs.zip)
    'src_dir': 'dtm_src',              # where it gets unzipped
    'zmin': ZMIN, 'zmax': TOPO_ZMAX,
    'valid_range_m': (-6000, 2000),    # heights outside this are treated as bad pixels (Gale spans about -4,700 to +350 m)
    'azimuth': 315, 'altitude': 45,    # sun direction (degrees clockwise from north) and height above the horizon
    'z_factor': 2.0,                   # vertical exaggeration for the shading
    'save_dem': True,                  # also write out/dem/gale_dem_20m.tif (float32 metres) for the model features
}
MARS_RADIUS_M = 3396190.0
MARS_CRS_PROJ4 = '+proj=longlat +R=3396190 +no_defs'
PREVIEW_Z = 9                          # zoom used for the preview images in cell 12
print('box', BBOX, '| imagery zooms', ZMIN, '-', IMAGERY_ZMAX, '| topo zooms', ZMIN, '-', TOPO_ZMAX)

box (136.75, -6.57, 139.24, -4.15) | imagery zooms 7 - 12 | topo zooms 7 - 11


In [8]:
NS = {'w': 'http://www.opengis.net/wmts/1.0', 'ows': 'http://www.opengis.net/ows/1.1'}

def fetch(url, tries=3):
    err = b''
    for i in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'gale-tiles/1.0 (Space Apps 2026)'})
            with urllib.request.urlopen(req, timeout=30) as r:
                return r.status, r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return 404, b''
            err = str(e).encode()
        except Exception as e:
            err = str(e).encode()
        time.sleep(1.5 * (i + 1))
    return 0, err

def capabilities(product):
    url = f'{BASE}/{product}/1.0.0/WMTSCapabilities.xml'
    status, body = fetch(url)
    if status != 200:
        raise RuntimeError(f'GetCapabilities failed ({status}) for {url}: {body[:200]}')
    root = ET.fromstring(body)
    layer = root.find('.//w:Layer', NS)
    fmt = layer.find('w:Format', NS).text
    return {
        'style': layer.find('w:Style/ows:Identifier', NS).text,
        'tms': layer.find('w:TileMatrixSetLink/w:TileMatrixSet', NS).text,
        'format': fmt,
        'ext': 'jpg' if 'jpeg' in fmt else 'png',
        'bbox': layer.find('ows:WGS84BoundingBox/ows:LowerCorner', NS).text + ' ' + layer.find('ows:WGS84BoundingBox/ows:UpperCorner', NS).text,
        'zooms': [int(m.find('ows:Identifier', NS).text) for m in root.findall('.//w:TileMatrix', NS)],
    }

def tile_range(bbox, z):
    # inclusive (col_min, col_max, row_min, row_max) of the tiles covering bbox at zoom z
    lon0, lat0, lon1, lat1 = bbox
    n_cols, n_rows = 2 ** (z + 1), 2 ** z
    deg = 180.0 / 2 ** z
    clamp = lambda v, hi: max(0, min(hi, v))
    c0 = clamp(int(math.floor((lon0 + 180.0) / deg)), n_cols - 1)
    c1 = clamp(int(math.floor((lon1 + 180.0) / deg)), n_cols - 1)
    r0 = clamp(int(math.floor((90.0 - lat1) / deg)), n_rows - 1)
    r1 = clamp(int(math.floor((90.0 - lat0) / deg)), n_rows - 1)
    return c0, c1, r0, r1

def tile_bounds(z, row, col):
    # (lon_min, lat_min, lon_max, lat_max) of one tile
    deg = 180.0 / 2 ** z
    return (-180 + col * deg, 90 - (row + 1) * deg, -180 + (col + 1) * deg, 90 - row * deg)

def tile_for(lon, lat, z):
    deg = 180.0 / 2 ** z
    return int((90 - lat) // deg), int((lon + 180) // deg)       # row, col

# Self-check: Gale's centre (137.8 E, 5.4 S) must fall inside the tile range at every zoom
for z in range(ZMIN, IMAGERY_ZMAX + 1):
    c0, c1, r0, r1 = tile_range(BBOX, z)
    r, c = tile_for(137.8, -5.4, z)
    assert c0 <= c <= c1 and r0 <= r <= r1, z
    lo, la0, hi, la1 = tile_bounds(z, r, c)
    assert lo <= 137.8 <= hi and la0 <= -5.4 <= la1, z
print('tile math ok')

tile math ok


In [9]:
# Confirm the Trek product's real extension, style, tile matrix set and max zoom before downloading
CAPS = {}
for name, cfg in LAYERS.items():
    CAPS[name] = capabilities(cfg['product'])
    cfg['zmax'] = min(cfg['zmax'], max(CAPS[name]['zooms']))
    print(name, cfg['product'])
    print(json.dumps(CAPS[name], indent=2))

imagery Gale_CTX_BlockAdj_dd
{
  "style": "default",
  "tms": "default028mm",
  "format": "image/png",
  "ext": "png",
  "bbox": "135.1842593 -6.5732988 139.2413012 -2.8077809",
  "zooms": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12
  ]
}


In [10]:
# Dry run: tiles per zoom for each layer, pixel size, and memory for the DTM mosaic
M_PER_DEG = 2 * math.pi * MARS_RADIUS_M / 360.0
grand = 0
for name, zmin, zmax in (('imagery', LAYERS['imagery']['zmin'], LAYERS['imagery']['zmax']),
                         ('topo', TOPO['zmin'], TOPO['zmax'])):
    total = 0
    print(name)
    for z in range(zmin, zmax + 1):
        c0, c1, r0, r1 = tile_range(BBOX, z)
        n = (c1 - c0 + 1) * (r1 - r0 + 1)
        total += n
        m_px = 180.0 / 2 ** z / 256 * M_PER_DEG
        print(f'  z{z}: cols {c0}-{c1}, rows {r0}-{r1} -> {n:5d} tiles, {m_px:6.1f} m/pixel')
    print(f'  total: {total} tiles\n')
    grand += total
print(f'both layers: {grand} tiles (the topo count is an upper bound: fully empty tiles are skipped)')

c0, c1, r0, r1 = tile_range(BBOX, TOPO['zmax'])
W, H = (c1 - c0 + 1) * 256, (r1 - r0 + 1) * 256
print(f'\nDTM mosaic at z{TOPO["zmax"]}: {W} x {H} px = {W * H / 1e6:.0f} M pixels')
print(f'memory: float32 DEM {W * H * 4 / 1e6:.0f} MB + grey/alpha image {W * H * 2 / 1e6:.0f} MB (plus working space, so allow ~1 GB)')

imagery
  z7: cols 225-227, rows 66-68 ->     9 tiles,  325.6 m/pixel
  z8: cols 450-454, rows 133-137 ->    25 tiles,  162.8 m/pixel
  z9: cols 900-908, rows 267-274 ->    72 tiles,   81.4 m/pixel
  z10: cols 1801-1816, rows 535-549 ->   240 tiles,   40.7 m/pixel
  z11: cols 3603-3632, rows 1071-1098 ->   840 tiles,   20.4 m/pixel
  z12: cols 7207-7264, rows 2142-2197 ->  3248 tiles,   10.2 m/pixel
  total: 4434 tiles

topo
  z7: cols 225-227, rows 66-68 ->     9 tiles,  325.6 m/pixel
  z8: cols 450-454, rows 133-137 ->    25 tiles,  162.8 m/pixel
  z9: cols 900-908, rows 267-274 ->    72 tiles,   81.4 m/pixel
  z10: cols 1801-1816, rows 535-549 ->   240 tiles,   40.7 m/pixel
  z11: cols 3603-3632, rows 1071-1098 ->   840 tiles,   20.4 m/pixel
  total: 1186 tiles

both layers: 5620 tiles (the topo count is an upper bound: fully empty tiles are skipped)

DTM mosaic at z11: 7680 x 7168 px = 55 M pixels
memory: float32 DEM 220 MB + grey/alpha image 110 MB (plus working space, so allow ~1

In [11]:
def download_layer(name, cfg, workers=6):
    cap = CAPS[name]
    bbox = cfg.get('bbox', BBOX)
    root = os.path.join(OUT, name)
    jobs, per_zoom = [], {}
    for z in range(cfg['zmin'], cfg['zmax'] + 1):
        c0, c1, r0, r1 = tile_range(bbox, z)
        per_zoom[z] = {'cols': [c0, c1], 'rows': [r0, r1]}
        jobs += [(z, r, c) for r in range(r0, r1 + 1) for c in range(c0, c1 + 1)]

    def work(job):
        z, r, c = job
        path = os.path.join(root, str(z), str(r), f"{c}.{cap['ext']}")
        if os.path.exists(path):
            return (z, 'cached')
        url = f"{BASE}/{cfg['product']}/1.0.0/{cap['style']}/{cap['tms']}/{z}/{r}/{c}.{cap['ext']}"
        status, body = fetch(url)
        if status == 200:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            with open(path, 'wb') as f:
                f.write(body)
            return (z, 'ok')
        return (z, 'no_coverage' if status == 404 else f'error_{status}')

    with ThreadPoolExecutor(workers) as ex:
        results = list(ex.map(work, jobs))
    summary = {}
    for z, res in results:
        summary.setdefault(res, 0)
        summary[res] += 1
    by_zoom = {}
    for z, res in results:
        by_zoom.setdefault(z, {}).setdefault(res, 0)
        by_zoom[z][res] += 1

    os.makedirs(root, exist_ok=True)
    with open(os.path.join(root, 'manifest.json'), 'w') as f:
        json.dump({
            'layer': name, 'product': cfg['product'],
            'source': f"{BASE}/{cfg['product']}/1.0.0/",
            'retrieved': time.strftime('%Y-%m-%d'),
            'bbox_lonlat': bbox, 'ext': cap['ext'], 'tile_size': 256,
            'scheme': 'WMTS geographic grid, row 0 at north, zoom 0 = 2x1 tiles',
            'tile_matrix_set': cap['tms'], 'zoom_range': [cfg['zmin'], cfg['zmax']],
            'per_zoom': per_zoom, 'summary': summary, 'summary_by_zoom': by_zoom,
            'attribution': 'Image: NASA/JPL-Caltech/MSSS (MRO CTX), via NASA/JPL Solar System Treks',
        }, f, indent=2)
    return summary, by_zoom

for name, cfg in LAYERS.items():
    summary, by_zoom = download_layer(name, cfg)
    print(name, summary)
    for z in sorted(by_zoom):
        print('  z%d' % z, by_zoom[z])
# Re-run this cell to retry failures; finished tiles are skipped.
# If a low zoom (z7/z8) is all 'no_coverage', Trek has no tiles there: set ZMIN = 9 in cell 2 and rerun from cell 2.
# z12 is the biggest download (about 4x z11). If it is too slow or too large, set IMAGERY_ZMAX = 11 and rerun from cell 2.

imagery {'cached': 4434}
  z7 {'cached': 9}
  z8 {'cached': 25}
  z9 {'cached': 72}
  z10 {'cached': 240}
  z11 {'cached': 840}
  z12 {'cached': 3248}


## Part 2: the topographic layer, built from the DTMs

Mars Trek serves **display images**, not elevation values, and CTX is a photo mosaic, so the topographic layer is made from real heights instead:

1. **Unzip and inspect** `DTMs.zip`: one 20 m/pixel GeoTIFF per CTX stereo pair (cell 8).
2. **Mosaic** them onto the z11 tile grid in a Mars geographic CRS, so one 256 x 256 block of the mosaic is exactly one z11 tile (cell 9). Where DTMs overlap, the first file (by name) wins.
3. **Check coverage** inside the box. This is where the box estimate gets confirmed or corrected.
4. **Hillshade** from the heights (cell 10): grey shading only, no colors, because the UI team adds those. Shading uses meters per pixel for Mars (radius 3,396,190 m), not Earth.
5. **Cut into tiles** at z11, then build each lower zoom by shrinking 2 x 2 child tiles into one parent (cell 11). All tiles are grey + transparency PNGs (no-data areas are transparent).

Caveats from the dataset authors: each DTM is tied to MOLA but **nothing is adjusted across DTM boundaries**, so small height steps can appear at seams (they show up as faint lines in a hillshade). Check the quality (FOM) layers and look at the preview before shipping. Credit: "DTM: NASA/JPL-Caltech/MSSS/NHM" and "Image: NASA/JPL-Caltech/MSSS".

Where the file comes from: [figshare 6584984](https://figshare.com/articles/Gale_Crater_Stereo_CTX_DTMs_Orthoimages/6584984), file `DTMs.zip` (about 355 MiB, CC BY 4.0). `Index_ DTMs.xlsx` and `Gale_CTX_DTMs map.png` on the same page show which pairs exist.

In [12]:
import numpy as np
import rasterio
from affine import Affine
from rasterio.warp import transform_bounds, reproject, Resampling

DST_CRS = rasterio.crs.CRS.from_proj4(MARS_CRS_PROJ4)

assert os.path.exists(TOPO['zip']), f"{TOPO['zip']} not found next to the notebook. Download it from the figshare page above."
os.makedirs(TOPO['src_dir'], exist_ok=True)
root_dir = os.path.realpath(TOPO['src_dir'])
with zipfile.ZipFile(TOPO['zip']) as zf:
    for member in zf.namelist():                      # refuse paths that would escape the folder
        assert os.path.realpath(os.path.join(root_dir, member)).startswith(root_dir + os.sep) or member.endswith('/'), member
    zf.extractall(root_dir)

dtm_files = sorted(p for p in glob.glob(os.path.join(root_dir, '**', '*'), recursive=True)
                   if p.lower().endswith(('.tif', '.tiff')) and 'fom' not in os.path.basename(p).lower())
print(len(dtm_files), 'GeoTIFFs found')

print(f"{'file':40s} {'size':>13s} {'res (m)':>8s} {'nodata':>10s}  lon_min  lat_min  lon_max  lat_max  in box?")
for p in dtm_files:
    with rasterio.open(p) as s:
        if s.crs is None:
            print(os.path.basename(p), 'has no CRS: cannot be placed, skipping')
            continue
        l, b, r, t = transform_bounds(s.crs, DST_CRS, *s.bounds, densify_pts=21)
        hit = not (r < BBOX[0] or l > BBOX[2] or t < BBOX[1] or b > BBOX[3])
        print(f"{os.path.basename(p)[:40]:40s} {s.width:>6d}x{s.height:<6d} {abs(s.res[0]):8.1f} {str(s.nodata):>10s} {l:8.3f} {b:8.3f} {r:8.3f} {t:8.3f}  {'yes' if hit else 'no'}")
print('\nCRS of the first file:', rasterio.open(dtm_files[0]).crs.to_string()[:120])

8 GeoTIFFs found
file                                              size  res (m)     nodata  lon_min  lat_min  lon_max  lat_max  in box?
PG_Gale_CTX_1_AATE_20m.tif                 2396x9447       20.2 -3.4028230607370965e+38  136.365   -7.024  137.183   -3.799  yes
PG_Gale_CTX_2_2_AATE_20m.tif               2191x8227       20.2 -3.4028230607370965e+38  136.751   -6.877  137.498   -4.069  yes
PG_Gale_CTX_3_3_AATE_20m.tif               2394x9491       20.2 -3.4028230607370965e+38  137.048   -6.859  137.865   -3.619  yes
PG_Gale_CTX_4_AATE_20m.tif                 2258x8540       20.2 -3.4028230607370965e+38  137.395   -6.783  138.166   -3.868  yes
PG_Gale_CTX_5_AATE_20m.tif                 2269x8434       20.2 -3.4028230607370965e+38  137.730   -6.759  138.505   -3.880  yes
PG_Gale_CTX_6_AATE_20m.tif                 2245x8446       20.2 -3.4028230607370965e+38  138.114   -6.775  138.880   -3.892  yes
PG_Gale_CTX_7_2_AATE_20m.tif               2324x8839       20.2 -3.4028230607370965e+38  

In [13]:
# Mosaic the DTMs onto the zmax tile grid (so 256 x 256 blocks of the mosaic are exactly tiles)
zmax = TOPO['zmax']
deg_tile = 180.0 / 2 ** zmax
px = deg_tile / 256                                   # pixel size in degrees
c0, c1, r0, r1 = tile_range(BBOX, zmax)
west, north = -180 + c0 * deg_tile, 90 - r0 * deg_tile
W, H = (c1 - c0 + 1) * 256, (r1 - r0 + 1) * 256
dst_transform = Affine(px, 0, west, 0, -px, north)

dem = np.full((H, W), np.nan, np.float32)
lo, hi = TOPO['valid_range_m']
used = []
for p in dtm_files:
    with rasterio.open(p) as s:
        if s.crs is None:
            continue
        l, b, r, t = transform_bounds(s.crs, DST_CRS, *s.bounds, densify_pts=21)
        ca, cb = max(0, int(math.floor((l - west) / px))), min(W, int(math.ceil((r - west) / px)))
        ra, rb = max(0, int(math.floor((north - t) / px))), min(H, int(math.ceil((north - b) / px)))
        if cb <= ca or rb <= ra:
            continue                                  # this DTM does not touch the box
        tmp = np.full((rb - ra, cb - ca), np.nan, np.float32)
        reproject(source=rasterio.band(s, 1), destination=tmp,
                  dst_transform=dst_transform * Affine.translation(ca, ra), dst_crs=DST_CRS,
                  src_nodata=s.nodata, dst_nodata=np.nan, resampling=Resampling.bilinear)
        tmp[(tmp < lo) | (tmp > hi)] = np.nan         # drop nodata leftovers and impossible heights
        region = dem[ra:rb, ca:cb]
        take = np.isnan(region) & np.isfinite(tmp)    # first valid file wins
        region[take] = tmp[take]
        used.append((os.path.basename(p), int(take.sum())))
print(len(used), 'DTMs contributed pixels:')
for name, n in used:
    print(f'  {name[:50]:50s} {n:>10d} px')

# Coverage check inside the box
bx0, bx1 = int((BBOX[0] - west) / px), int(math.ceil((BBOX[2] - west) / px))
by0, by1 = int((north - BBOX[3]) / px), int(math.ceil((north - BBOX[1]) / px))
inside = dem[by0:by1, bx0:bx1]
valid = float(np.isfinite(inside).mean())
print(f'\nvalid DEM pixels inside the box: {valid:.2%}')
if valid < 0.995:
    ys, xs = np.where(~np.isfinite(inside))
    print('WARNING: holes inside the box. Bounding box of the holes: '
          f'lon {west + (bx0 + xs.min()) * px:.3f} to {west + (bx0 + xs.max()) * px:.3f}, '
          f'lat {north - (by0 + ys.max()) * px:.3f} to {north - (by0 + ys.min()) * px:.3f}. '
          'Shrink BBOX in cell 2 (and rerun from there) or accept transparent holes.')
print('height range (m): %.0f to %.0f' % (np.nanmin(dem), np.nanmax(dem)))

if TOPO['save_dem']:
    os.makedirs(os.path.join(OUT, 'dem'), exist_ok=True)
    with rasterio.open(os.path.join(OUT, 'dem', 'gale_dem_20m.tif'), 'w', driver='GTiff', height=H, width=W, count=1,
                       dtype='float32', crs=DST_CRS, transform=dst_transform, nodata=np.nan,
                       compress='deflate', tiled=True) as d:
        d.write(dem, 1)
    print('wrote', os.path.join(OUT, 'dem', 'gale_dem_20m.tif'))

8 DTMs contributed pixels:
  PG_Gale_CTX_1_AATE_20m.tif                            6590956 px
  PG_Gale_CTX_2_2_AATE_20m.tif                          6911189 px
  PG_Gale_CTX_3_3_AATE_20m.tif                          7893405 px
  PG_Gale_CTX_4_AATE_20m.tif                            6329227 px
  PG_Gale_CTX_5_AATE_20m.tif                            7159508 px
  PG_Gale_CTX_6_AATE_20m.tif                            7705412 px
  PG_Gale_CTX_7_2_AATE_20m.tif                          7098184 px
  PG_Gale_CTX_8_AATE_20m.tif                            5311587 px

valid DEM pixels inside the box: 100.00%
height range (m): -4720 to 1120
wrote out_tiles\dem\gale_dem_20m.tif


In [14]:
import numpy as np

M_PER_DEG = 2 * math.pi * MARS_RADIUS_M / 360.0

def shade_block(block, px_deg, north_deg, azimuth, altitude, z_factor):
    # Hillshade 0..1 for a block of heights (rows run north to south). Pixel size in metres is Mars-based and
    # the east-west spacing shrinks with cos(latitude).
    h = block.shape[0]
    dy = px_deg * M_PER_DEG
    lat = north_deg - (np.arange(h) + 0.5) * px_deg
    dx = dy * np.cos(np.radians(lat))[:, None]
    gy, gx = np.gradient(block)                       # per pixel: rows (south is +), columns (east is +)
    dzdx = gx / dx                                    # rise per metre toward the east
    dzdy = -gy / dy                                   # rise per metre toward the north
    az, alt = math.radians(azimuth), math.radians(altitude)
    lx, ly, lz = math.sin(az) * math.cos(alt), math.cos(az) * math.cos(alt), math.sin(alt)   # unit vector toward the sun
    shade = (-z_factor * dzdx * lx - z_factor * dzdy * ly + lz) / np.sqrt(1 + (z_factor ** 2) * (dzdx ** 2 + dzdy ** 2))
    return np.clip(shade, 0, 1)

def render_hillshade(dem, px_deg, north_deg, cfg, chunk=1024):
    # Grey hillshade as an (H, W, 2) uint8 array: [..., 0] = grey (flat ground is about 180), [..., 1] = alpha
    # (0 where there is no height data). No colors: the UI team adds those. Done in row chunks (with a
    # 1-row halo) to save memory.
    H, W = dem.shape
    out = np.zeros((H, W, 2), np.uint8)
    for r in range(0, H, chunk):
        a, b = max(0, r - 1), min(H, r + chunk + 1)
        shade = shade_block(dem[a:b], px_deg, north_deg - a * px_deg, cfg['azimuth'], cfg['altitude'], cfg['z_factor'])
        s0, s1 = r - a, r - a + (min(H, r + chunk) - r)
        shade = shade[s0:s1]
        out[r:r + (s1 - s0), :, 0] = np.clip(np.nan_to_num(shade) * 255, 0, 255).astype(np.uint8)
        out[r:r + (s1 - s0), :, 1] = np.where(np.isfinite(shade), 255, 0)
    return out

elev_min, elev_max = float(np.nanmin(dem)), float(np.nanmax(dem))
shaded = render_hillshade(dem, px, north, TOPO)
print(f'heights in the mosaic: {elev_min:.0f} to {elev_max:.0f} m')
print('rendered', shaded.shape, 'visible pixels: %.1f%%' % (100 * (shaded[..., 1] > 0).mean()))

heights in the mosaic: -4720 to 1120 m
rendered (7168, 7680, 2) visible pixels: 99.9%


In [15]:
from PIL import Image

topo_root = os.path.join(OUT, 'topo')
zmin, zmax = TOPO['zmin'], TOPO['zmax']

def tile_path(z, r, c):
    return os.path.join(topo_root, str(z), str(r), f'{c}.png')

# 1) the top zoom: cut the mosaic into 256 x 256 tiles (grey + alpha PNG)
per_zoom, counts = {}, {}
c0, c1, r0, r1 = tile_range(BBOX, zmax)
n = 0
for r in range(r0, r1 + 1):
    for c in range(c0, c1 + 1):
        block = shaded[(r - r0) * 256:(r - r0 + 1) * 256, (c - c0) * 256:(c - c0 + 1) * 256]
        if not block[..., 1].any():
            continue                                  # fully empty tile: skip
        os.makedirs(os.path.dirname(tile_path(zmax, r, c)), exist_ok=True)
        Image.fromarray(block).save(tile_path(zmax, r, c), optimize=True)
        n += 1
per_zoom[zmax] = {'cols': [c0, c1], 'rows': [r0, r1]}
counts[zmax] = n
print(f'z{zmax}: {n} tiles')

# 2) lower zooms: each parent tile is its 2 x 2 children shrunk to half size
for z in range(zmax - 1, zmin - 1, -1):
    c0, c1, r0, r1 = tile_range(BBOX, z)
    n = 0
    for r in range(r0, r1 + 1):
        for c in range(c0, c1 + 1):
            canvas, found = Image.new('LA', (512, 512), (0, 0)), False
            for dr in (0, 1):
                for dc in (0, 1):
                    p = tile_path(z + 1, 2 * r + dr, 2 * c + dc)
                    if os.path.exists(p):
                        canvas.paste(Image.open(p).convert('LA'), (dc * 256, dr * 256))
                        found = True
            if not found:
                continue
            os.makedirs(os.path.dirname(tile_path(z, r, c)), exist_ok=True)
            canvas.resize((256, 256), Image.Resampling.LANCZOS).save(tile_path(z, r, c), optimize=True)
            n += 1
    per_zoom[z] = {'cols': [c0, c1], 'rows': [r0, r1]}
    counts[z] = n
    print(f'z{z}: {n} tiles')

size_mb = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(topo_root) for f in fs) / 1e6
with open(os.path.join(topo_root, 'manifest.json'), 'w') as f:
    json.dump({
        'layer': 'topo', 'product': 'Gale Crater stereo CTX DTMs (20 m/pixel), grey hillshade (no colors), built locally',
        'source': 'https://figshare.com/articles/Gale_Crater_Stereo_CTX_DTMs_Orthoimages/6584984',
        'retrieved': time.strftime('%Y-%m-%d'), 'bbox_lonlat': BBOX, 'ext': 'png', 'image_mode': 'grey + alpha',
        'tile_size': 256, 'scheme': 'WMTS geographic grid, row 0 at north, zoom 0 = 2x1 tiles (same grid as imagery)',
        'zoom_range': [zmin, zmax], 'per_zoom': per_zoom, 'tiles_written': counts,
        'sun_azimuth': TOPO['azimuth'], 'sun_altitude': TOPO['altitude'], 'z_factor': TOPO['z_factor'],
        'elevation_range_m': [elev_min, elev_max],
        'note': 'Grey shading only. The shading does not encode height; colors are added by the UI team.',
        'attribution': 'DTM: NASA/JPL-Caltech/MSSS/NHM (Grindrod & Davis, CC BY 4.0)',
    }, f, indent=2)
print(f'\n{sum(counts.values())} tiles, {size_mb:.0f} MB in {topo_root}')

z11: 840 tiles
z10: 240 tiles
z9: 72 tiles
z8: 25 tiles
z7: 9 tiles

1186 tiles, 61 MB in out_tiles\topo


In [ ]:
# Stitch both layers at every zoom from PREVIEW_ZMIN to PREVIEW_ZMAX into one image each, to eyeball coverage, seams and alignment (needs: pillow)
from PIL import Image

PREVIEW_ZMIN, PREVIEW_ZMAX = 11, 11
SHOW_MAX_PX = 1200          # the notebook shows a shrunken copy; the full-size PNG is saved to out/

def stitch(name, z):
    m = json.load(open(os.path.join(OUT, name, 'manifest.json')))
    pz = m['per_zoom'].get(str(z))
    if pz is None:
        print(name, 'has no zoom', z)
        return None
    c0, c1 = pz['cols']
    r0, r1 = pz['rows']
    canvas = Image.new('RGB', ((c1 - c0 + 1) * 256, (r1 - r0 + 1) * 256), (30, 30, 30))
    for r in range(r0, r1 + 1):
        for c in range(c0, c1 + 1):
            p = os.path.join(OUT, name, str(z), str(r), f"{c}.{m['ext']}")
            if os.path.exists(p):
                im = Image.open(p).convert('RGBA')
                canvas.paste(im, ((c - c0) * 256, (r - r0) * 256), im)
    canvas.save(os.path.join(OUT, f'{name}_preview_z{z}.png'))
    return canvas

for name in ('imagery', 'topo'):
    if not os.path.exists(os.path.join(OUT, name, 'manifest.json')):
        continue
    for z in range(PREVIEW_ZMIN, PREVIEW_ZMAX + 1):
        img = stitch(name, z)
        if img is None:
            continue
        print(f'{name} z{z}: {img.width} x {img.height} px -> out/{name}_preview_z{z}.png')
        shown = img.copy()
        shown.thumbnail((SHOW_MAX_PX, SHOW_MAX_PX))
        #display(shown)

imagery z11: 7680 x 7168 px -> out/imagery_preview_z11.png
topo z11: 7680 x 7168 px -> out/topo_preview_z11.png


: 

In [ ]:
# Elevation, slope and aspect from the DTM mosaic (cell 9), saved as three aligned GeoTIFFs in out/dem/
#   elevation  gale_dem_20m.tif     metres above the Mars datum
#   slope      gale_slope_20m.tif   degrees, 0 = flat, 90 = vertical
#   aspect     gale_aspect_20m.tif  degrees clockwise from north (0 = N, 90 = E, 180 = S, 270 = W) of the
#                                   DOWNHILL direction, i.e. the way the slope faces. Empty (NaN) where the ground is exactly flat.
# All three are float32, NaN = no data, same grid and CRS as the DEM, so pixel (row, col) is the same place in each.
import numpy as np

DEM_DIR = os.path.join(OUT, 'dem')
os.makedirs(DEM_DIR, exist_ok=True)
ELEV_TIF = os.path.join(DEM_DIR, 'gale_dem_20m.tif')
SLOPE_TIF = os.path.join(DEM_DIR, 'gale_slope_20m.tif')
ASPECT_TIF = os.path.join(DEM_DIR, 'gale_aspect_20m.tif')

def slope_aspect(padded, px_deg, north_deg, row0):
    # Horn's 3 x 3 method (the one gdaldem uses). `padded` is the DEM with a 1-pixel border, so the interior rows
    # are the DEM rows row0 .. row0 + n - 1. East-west pixel size in metres shrinks with cos(latitude).
    z = padded.astype(np.float64)
    n = z.shape[0] - 2
    dy = px_deg * M_PER_DEG
    lat = north_deg - (row0 + np.arange(n) + 0.5) * px_deg
    dx = dy * np.cos(np.radians(lat))[:, None]
    a, b, c = z[:-2, :-2], z[:-2, 1:-1], z[:-2, 2:]
    d, f = z[1:-1, :-2], z[1:-1, 2:]
    g, h, i = z[2:, :-2], z[2:, 1:-1], z[2:, 2:]
    dzdx = ((c + 2 * f + i) - (a + 2 * d + g)) / (8 * dx)      # rise per metre toward the east
    dzdy = ((a + 2 * b + c) - (g + 2 * h + i)) / (8 * dy)      # rise per metre toward the north
    slope = np.degrees(np.arctan(np.hypot(dzdx, dzdy)))
    aspect = np.degrees(np.arctan2(-dzdx, -dzdy)) % 360.0     # downhill = minus the gradient; atan2(east, north)
    aspect[(dzdx == 0) & (dzdy == 0)] = np.nan
    return slope.astype(np.float32), aspect.astype(np.float32)

def terrain_chunks(dem, px_deg, north_deg, chunk=512):
    # Yields (first_row, slope, aspect) in row chunks so the full-size temporaries never exist at once.
    padded = np.pad(dem, 1, constant_values=np.nan)
    for r in range(0, dem.shape[0], chunk):
        n = min(chunk, dem.shape[0] - r)
        s, a = slope_aspect(padded[r:r + n + 2], px_deg, north_deg, r)
        yield r, s, a

profile = dict(driver='GTiff', height=H, width=W, count=1, dtype='float32', crs=DST_CRS,
               transform=dst_transform, nodata=np.nan, compress='deflate', tiled=True)
from rasterio.windows import Window

if not os.path.exists(ELEV_TIF):                              # cell 9 only saves it when TOPO['save_dem'] is True
    with rasterio.open(ELEV_TIF, 'w', **profile) as d:
        d.write(dem, 1)

smin = amin = np.inf
n_valid = n_steep15 = n_steep30 = 0
smax = -np.inf
with rasterio.open(SLOPE_TIF, 'w', **profile) as ds, rasterio.open(ASPECT_TIF, 'w', **profile) as da:
    for r, s, a in terrain_chunks(dem, px, north):
        win = Window(0, r, W, s.shape[0])
        ds.write(s, 1, window=win)
        da.write(a, 1, window=win)
        ok = np.isfinite(s)
        n_valid += int(ok.sum())
        n_steep15 += int((s[ok] > 15).sum())
        n_steep30 += int((s[ok] > 30).sum())
        if ok.any():
            smax = max(smax, float(s[ok].max()))
print('wrote', ELEV_TIF, SLOPE_TIF, ASPECT_TIF, sep='\n  ')
print(f'pixels with slope: {n_valid:,}   steeper than 15 deg: {n_steep15 / n_valid:.1%}   '
      f'steeper than 30 deg: {n_steep30 / n_valid:.1%}   steepest: {smax:.1f} deg')

def terrain_at(points):
    # Elevation (m), slope (deg) and aspect (deg from north) at (lon, lat) points, e.g. the rover traverse.
    # Points outside the grid or on no-data pixels give NaN.
    rows = []
    with rasterio.open(ELEV_TIF) as e, rasterio.open(SLOPE_TIF) as s, rasterio.open(ASPECT_TIF) as a:
        for lon, lat in points:
            try:
                rows.append((lon, lat, *(float(next(src.sample([(lon, lat)]))[0]) for src in (e, s, a))))
            except Exception:
                rows.append((lon, lat, np.nan, np.nan, np.nan))
    return rows

print('\nBradbury Landing (137.44 E, 4.59 S), the Gale centre and a crater-floor point:')
print('   lon      lat   elev_m  slope_deg  aspect_deg')
for lon, lat, el, sl, asp in terrain_at([(137.44, -4.59), (137.80, -5.40), (137.0, -5.0)]):
    print(f'{lon:7.2f} {lat:8.2f} {el:8.0f} {sl:10.1f} {asp:11.0f}')

## Handoff to the UI/UX team

Give them the `out/` folder (`out/imagery/` and `out/topo/`, each with a `manifest.json`; `out/dem/` holds elevation, slope and aspect rasters for the data pipeline, not the map). Serve it from any static host.

**Layers**
- **Base:** CTX surface imagery (`imagery`, PNG, grayscale photo mosaic).
- **Toggle on top:** grey hillshaded topography (`topo`, grey + transparency PNG). It has **no colors**: the UI team adds those. Give it an opacity slider so the photo detail shows through. One option for tinting is CSS `mix-blend-mode: multiply` on the layer (set `className: 'topo-tiles'` as below), so whatever color sits underneath shows through the shading. That has not been tested.
- **Above those later:** heatmap, Curiosity traverse and routes, landmark markers.
- The shading does not encode height, so a height-based color scheme needs the DEM (`out/dem/gale_dem_20m.tif`), not these tiles. The height range of the area is in `out/topo/manifest.json` (`elevation_range_m`) for a legend.

**Leaflet.** The grid is geographic, so use `L.CRS.EPSG4326`. In Leaflet 1.x that CRS has 2 x 1 tiles at zoom 0 (checked in the Leaflet 1.9.4 source), which is the same as the Trek grid, so **the Leaflet zoom equals the folder zoom** and no zoom offset is needed. This page has not been run in a browser, so check that the imagery lines up with a known feature first.
```js
const BOUNDS = [[-6.57, 136.75], [-4.15, 139.24]];            // [[south, west], [north, east]]
const map = L.map('map', { crs: L.CRS.EPSG4326, center: [-5.4, 137.8], zoom: 8,
                           minZoom: 7, maxZoom: 13, maxBounds: BOUNDS });

const imagery = L.tileLayer('tiles/imagery/{z}/{y}/{x}.png', {
  tileSize: 256, minNativeZoom: 7, maxNativeZoom: 12, bounds: BOUNDS,
  attribution: 'Image: NASA/JPL-Caltech/MSSS (MRO CTX), Mars Trek'
}).addTo(map);

const topo = L.tileLayer('tiles/topo/{z}/{y}/{x}.png', {
  tileSize: 256, minNativeZoom: 7, maxNativeZoom: 11, bounds: BOUNDS, opacity: 0.6, className: 'topo-tiles',
  attribution: 'DTM: NASA/JPL-Caltech/MSSS/NHM'
});

L.control.layers({ 'CTX imagery': imagery }, { 'Topography (hillshade)': topo }).addTo(map);
```
`{y}` is the row counted from the north and `{x}` the column from -180 degrees, which is how the folders are laid out (`z/row/col`).

Zoom 12 exists only for `imagery`. Because `topo` has `maxNativeZoom: 11`, Leaflet enlarges its z11 tiles at zoom 12 (and 13) instead of asking for tiles that do not exist. The shading therefore looks 2x softer than the photo at the deepest zoom, which is expected: the DTMs are 20 m/pixel.

**Before shipping**
- **Distances:** Leaflet's `distanceTo` and `map.distance` assume Earth's radius (6,371 km). Mars is 3,396 km, so Leaflet distances are about 1.88 times too long. Compute route lengths with Mars' radius (3,396,190 m) or the haversine formula.
- **MapLibre GL** only reads web-mercator tiles, so these geographic tiles will not line up there. Use Leaflet (above), or re-tile to web mercator.
- **Box edges:** the box clips the crater rim slightly on the west, south and north. Check the preview. If the UI needs the whole rim, the DTMs have gaps there, so the options are the 200 m USGS HRSC/MOLA DEM for a coarser fill, or a smaller crater-only view.
- **Seams:** the DTMs are not adjusted to each other, so faint lines can appear where two stereo pairs meet. Lower `z_factor` or the layer opacity if they distract.
- **Tile counts and size:** each extra zoom level is about 4x the tiles, and imagery z12 is the largest piece (cell 5 prints the counts). Cell 11 prints the size of `out/topo`; also check the size of `out/imagery` (Trek's CTX tiles are PNG) before hosting. If it is too big, lower `IMAGERY_ZMAX` or `TOPO_ZMAX` in cell 2.
- **Credits** for the provenance panel (requirements W11, N3): "Image: NASA/JPL-Caltech/MSSS" for CTX; "DTM: NASA/JPL-Caltech/MSSS/NHM" for the heights (CC BY 4.0). Check the Mars Trek terms for the imagery tiles.
- **For the temperature and route models:** `out/dem/` has three aligned 20 m GeoTIFFs (float32, NaN = no data, Mars geographic CRS): `gale_dem_20m.tif` (elevation, m), `gale_slope_20m.tif` (degrees) and `gale_aspect_20m.tif` (degrees clockwise from north, the downhill direction). Use these, not the shaded tiles. `terrain_at([(lon, lat), ...])` in the terrain cell reads all three at points such as the traverse. Insolation is not computed here.